# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions

- Step 1: Run the followings cell to get connected to your MongoDB. Make sure your credential is in dotenv file.
- Step 2: Put your answer inside each function. Please do not construct your own function.
- Step 3: You can test your function under test section.
- Step 4. Run test my function to confirm if my code is working.


### Connections

In [9]:
import os

from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

In [10]:
# Load environment variables from .env file
load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    raise ValueError(
        "❌ MONGODB_URI not found!\n"
        "Please create a .env file with your MongoDB credentials.\n"
        "See README.md for setup instructions."
    )
client = MongoClient(MONGODB_URI, server_api=ServerApi("1"))
# Send a ping to confirm a successful connection
try:
    client.admin.command("ping")
    print("✅ Successfully connected to MongoDB!")
except Exception as e:
    print(e)

✅ Successfully connected to MongoDB!


In [11]:
db = client.sample_mflix
movies = db.movies
print(f"📊 Database: {db.name}")
print(f"📁 Collection: movies ({movies.count_documents({})} documents)")

📊 Database: sample_mflix
📁 Collection: movies (21349 documents)



### Question 1

Question: From the `movies` collection, return the documents with the `plot` that starts with `"war"` in acending order of released date, print only title, plot and released fields. Limit the result to 5.


**Answer:**

In [26]:
# ============================================================================
# Question 1
# ============================================================================


import pymongo


for war_movie in movies.find({"plot": {"$regex": "^War"}}).sort('released', pymongo.ASCENDING).limit(5):
    print(f"{war_movie['title']} ({war_movie['released'].year}) - {war_movie['plot']}")

Nausicaè of the Valley of the Wind (1984) - Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.
Nausicaè of the Valley of the Wind (1984) - Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.
Heaven and Earth (1991) - Warlords Kagetora and Takeda each wish to prevent the other from gaining hegemony in feudal Japan. The two samurai leaders pursue one another across the countryside, engaging in massive ...
Under the Stars (2007) - Warning! This synopsis contains spoilers Bajo las estrellas (beneath the stars) features the selfish...
Aliens vs. Predator: Requiem (2007) - Warring alien and predator races descend on a small town, where unsuspecting residents must band together for any chance of survival.


### Question 2

Question: Group by `rated` and count the number of movies in each.

**Answer:**

In [17]:
# ============================================================================
# Question 2
# ============================================================================


# 1. Name the pipeline based on its intent (what it does)
group_by_rating_pipeline = [
    {
        "$group": {
            "_id": "$rated",            # Group by the 'rated' field
            "movie_count": {"$sum": 1}  # Count the number of movies in each group
        }
    },
    {
        "$sort": {"movie_count": pymongo.DESCENDING}  # Optional: Sort by the highest count first
    }
]

# 2. Name the result based on the data it holds (what it is)
movies_per_rating = movies.aggregate(group_by_rating_pipeline)

# 3. Name the loop variable based on what a single item represents
for rating_group in movies_per_rating:
    # Handle cases where a movie might not have a 'rated' field
    rating = rating_group['_id'] or 'Unrated'
    total = rating_group['movie_count']
    
    print(f"Rating: {rating} | Total Movies: {total}")



Rating: Unrated | Total Movies: 9894
Rating: R | Total Movies: 5537
Rating: PG-13 | Total Movies: 2321
Rating: PG | Total Movies: 1852
Rating: APPROVED | Total Movies: 709
Rating: G | Total Movies: 477
Rating: PASSED | Total Movies: 181
Rating: TV-14 | Total Movies: 89
Rating: TV-PG | Total Movies: 76
Rating: TV-MA | Total Movies: 60
Rating: TV-G | Total Movies: 59
Rating: GP | Total Movies: 44
Rating: M | Total Movies: 37
Rating: Approved | Total Movies: 5
Rating: TV-Y7 | Total Movies: 3
Rating: AO | Total Movies: 3
Rating: Not Rated | Total Movies: 1
Rating: OPEN | Total Movies: 1



### Question 3

Question: Count the number of movies with 3 comments or more.


**Answer:**


In [25]:
# ============================================================================
# Question 3
# ============================================================================

# build pipeline to count movies with 3 or more comments
comment_count = [
    {
        "$match": {"num_mflix_comments": {"$gte": 3}}    # Stage 1: Filter for movies with >= 3 comments
    },
    {
        "$count": "no_of_movies_>=3_comments"            # Stage 2: movie count of movies with >= 3 comments
    }
]

# Execute the pipeline
results = movies.aggregate(comment_count)

# Print the outcome
for outcome in results:
    print(f"Number of movies with 3 or more comments: {outcome['no_of_movies_>=3_comments']}")

Number of movies with 3 or more comments: 385
